# A spatial weight prior: kernels, training, and the tuning of every cell

Every unit of the input, the hidden layer(s) and the output sits at a position on a ring (or a
torus); the weight prior of each layer is a Gaussian kernel over those positions, so units that
are close in space get similar weights. The notebook sets the prior, trains a net under it on a
linear teacher ($x = Z W_X^\top$, $y = Z W_Y^\top$, $k$ latents $Z$), and looks at what
training produced: the tuning of each neuron to each latent, laid out by the neuron's position.

Set `CACHE_ROOT` in the first cell. The run is cached under it in a directory named by a hash of
the task, net, training and spatial settings, so re-running reloads a finished run, resumes an
interrupted one, and changing any setting starts a new one.

**Why the default sizes are small.** The prior's penalty matrix is the inverse of the kernel with
its eigenvalues floored at `MIN_REL_EIG` times the largest. A smooth kernel has eigenvalues down to
round-off, so the library default floor of 1e-6 gives a prior term of order 1e9 at the start, and
SGLD refuses to run unless `LR * WEIGHT_LAMBDA * lambda_max(M) < 1` for the stiffest direction of
the prior. With 16 hidden units on a ring, kernel length scales of 0.15 (about 2.4 neighbour
spacings) and a floor of 1e-2, that product is about 0.01 at `LR = 1e-2`, both optimisers are stable,
and 2000 epochs take seconds on a CPU. SGLD takes plain gradient steps plus noise, so in the same
number of epochs it fits the teacher less closely than Adam does. A larger network or a lower floor needs a smaller `LR`
(for SGLD) or `WEIGHT_LAMBDA`.

## Task and network

In [ ]:
CACHE_ROOT = None  # directory for the cached runs; None reads $FCT_CACHE_ROOT
# task: a linear teacher with m features and k latents
P = 256  # samples
M_IN = 12  # m = D_in: features of the input x and of the output y
K_LATENTS = 3  # k latents, fewer than m
DATA_SEED = 0
# network
N = 16  # hidden units per layer
L = 1  # hidden layers
PHI = "tanh"  # "tanh", "relu" or "sigmoid"
G = 1.0  # hidden-to-hidden gain
NET_SEED = 0
# loss weights
WEIGHT_LAMBDA = 3e-3  # multiplies the spatial weight penalty
RATE_LAMBDA = 1e-4  # multiplies the rate penalty
DEVICE = "cpu"  # "cpu", "cuda" or "auto"; these nets are tiny

## Training

In [ ]:
OPTIMIZER = "sgld"  # "sgld" (Langevin noise of variance 2 * LR / BETA per step) or "adam"
BETA = 1e4  # inverse temperature, read by SGLD only
LR = 1e-2
N_EPOCHS = 2000
TRAIN_SEED = 0

## Spatial prior

Embedding and kernel of each population, then the kernels and positions of every weight layer (`K_post (x) K_pre`).

In [ ]:
import matplotlib as mpl

from fct.cache import cache_root_from_env
from fct.report import show
from fct.spatial_report import (
    kernel_items,
    training_items,
    tuning_by_position_items,
    weight_items,
)
from fct.spatial_view import layer_spaces, make_config, train_run

mpl.rcParams["figure.dpi"] = 100
RUNS_ROOT = CACHE_ROOT or cache_root_from_env()

In [ ]:
from fct.spatial import SpatialConfig
from fct.spatial_view import population_spec

D = 1  # embedding dimension: a ring for 1, a torus for 2 (a grid then needs a shape, d <= 2)
BOX = 1.0  # side of the torus; length scales are in the same units
MIN_REL_EIG = 1e-2  # eigenvalue floor of the penalty matrices, relative to the largest
# one embedding and kernel per population: "grid" (evenly spaced) or "random" (uniform)
INPUT = population_spec("grid", length_scale=0.15, amplitude=1.0)
HIDDEN = population_spec("grid", length_scale=0.15, amplitude=1.0)  # shared by all hidden layers
OUTPUT = population_spec("grid", length_scale=0.15, amplitude=1.0)

SPATIAL = SpatialConfig(
    d=D, box=BOX, input=INPUT, hidden=HIDDEN, output=OUTPUT, min_rel_eig=MIN_REL_EIG
)
CFG = make_config(
    P=P,
    m=M_IN,
    k=K_LATENTS,
    N=N,
    L=L,
    phi=PHI,
    g=G,
    data_seed=DATA_SEED,
    net_seed=NET_SEED,
    weight_lambda=WEIGHT_LAMBDA,
    rate_lambda=RATE_LAMBDA,
    optimizer=OPTIMIZER,
    beta=BETA,
    lr=LR,
    n_epochs=N_EPOCHS,
    train_seed=TRAIN_SEED,
    spatial=SPATIAL,
    device=DEVICE,
)
SPACES = layer_spaces(CFG)
show(kernel_items(SPACES))

## Train

In [ ]:
RUN = train_run(CFG, RUNS_ROOT, log=print)
show([f"Run cached in `{RUN.cache_dir.name}`."])

## Loss, and the trained network's outputs

In [ ]:
show(training_items(RUN, CFG))

## First-layer weights over the embedding

In [ ]:
show(weight_items(RUN.net, SPACES))

## Tuning of every neuron to the latents

One panel per neuron, placed by its position in space (ring order along the rows for d = 1, its lattice cell for d = 2), one curve per latent: the neuron's mean rate against that latent, the other latents averaged out.

In [ ]:
N_BINS = 8  # rank bins of each latent and of the rates, as in the MIR
N_SHUFFLES = 5  # shuffles for the bias-corrected MIR in each title
MIR_KIND = "corrected"  # "corrected" or "raw"
N_COLS = 4  # panels per row on a ring; a plane lattice uses its own rows and columns

show(tuning_by_position_items(RUN, SPACES, N_BINS, MIR_KIND, N_SHUFFLES, TRAIN_SEED, N_COLS))